In [1]:
import pandas as pd

CodeHire is a tech recruitment platform. They have candidate data, job listings, and application records spread across multiple tables. Your job is to merge them, clean them up, and generate aggregated reports for the hiring team — all using pandas.


In [2]:
candidates = pd.DataFrame({
    "CandidateID": [1, 2, 3, 4, 5, 6],
    "Name": ["Aarav", "Bhavna", "Chirag", "Divya", "Eshan", "Fatima"],
    "City": ["Delhi", "Mumbai", "Bangalore", "Delhi", "Mumbai", "Bangalore"],
    "Skills": ["Python", "SQL", "Python", "ML", "Python", "SQL"],
    "YearsExp": [2, 5, 3, 7, 1, 4]
})

jobs = pd.DataFrame({
    "JobID": [101, 102, 103, 104],
    "Title": ["Data Analyst", "ML Engineer", "Python Dev", "SQL Specialist"],
    "MinExp": [2, 5, 2, 3],
    "Location": ["Delhi", "Bangalore", "Mumbai", "Delhi"]
})

applications = pd.DataFrame({
    "AppID": [1, 2, 3, 4, 5, 6, 7, 8],
    "CandidateID": [1, 2, 3, 1, 4, 5, 6, 3],
    "JobID": [101, 103, 102, 104, 101, 102, 103, 104],
    "Status": ["Selected", "Rejected", "Selected", "Pending",
               "Selected", "Rejected", "Pending", "Selected"],
    "InterviewScore": [88, 45, 92, 76, 85, 38, 67, 90]
})


# Part A — Merging

## 1: Merge applications with candidates on CandidateID — use an inner join

In [5]:
result = pd.merge(applications, candidates, on="CandidateID")
result

,AppID,CandidateID,JobID,Status,InterviewScore,Name,City,Skills,YearsExp
0,1,1,101,Selected,88,Aarav,Delhi,Python,2
1,2,2,103,Rejected,45,Bhavna,Mumbai,SQL,5
2,3,3,102,Selected,92,Chirag,Bangalore,Python,3
3,4,1,104,Pending,76,Aarav,Delhi,Python,2
4,5,4,101,Selected,85,Divya,Delhi,ML,7
5,6,5,102,Rejected,38,Eshan,Mumbai,Python,1
6,7,6,103,Pending,67,Fatima,Bangalore,SQL,4
7,8,3,104,Selected,90,Chirag,Bangalore,Python,3


## 2: Merge the result with jobs on JobID — use an inner join

In [6]:
df = pd.merge(result, jobs, on="JobID")
df

,AppID,CandidateID,JobID,Status,InterviewScore,Name,City,Skills,YearsExp,Title,MinExp,Location
0,1,1,101,Selected,88,Aarav,Delhi,Python,2,Data Analyst,2,Delhi
1,2,2,103,Rejected,45,Bhavna,Mumbai,SQL,5,Python Dev,2,Mumbai
2,3,3,102,Selected,92,Chirag,Bangalore,Python,3,ML Engineer,5,Bangalore
3,4,1,104,Pending,76,Aarav,Delhi,Python,2,SQL Specialist,3,Delhi
4,5,4,101,Selected,85,Divya,Delhi,ML,7,Data Analyst,2,Delhi
5,6,5,102,Rejected,38,Eshan,Mumbai,Python,1,ML Engineer,5,Bangalore
6,7,6,103,Pending,67,Fatima,Bangalore,SQL,4,Python Dev,2,Mumbai
7,8,3,104,Selected,90,Chirag,Bangalore,Python,3,SQL Specialist,3,Delhi


## 3: Store the final merged DataFrame as df — print its shape and columns

In [10]:
print(f"shape: {df.shape}")
print(f"columns: \n{df.columns}")

shape: (8, 12)
columns: 
Index(['AppID', 'CandidateID', 'JobID', 'Status', 'InterviewScore', 'Name',
       'City', 'Skills', 'YearsExp', 'Title', 'MinExp', 'Location'],
      dtype='object')


## 4: Now create a left join of candidates with applications on CandidateID — which candidates have never applied? Find them using isnull()

In [17]:
left_join = pd.merge(candidates, applications, on="CandidateID", how="left")
left_join

,CandidateID,Name,City,Skills,YearsExp,AppID,JobID,Status,InterviewScore
0,1,Aarav,Delhi,Python,2,1,101,Selected,88
1,1,Aarav,Delhi,Python,2,4,104,Pending,76
2,2,Bhavna,Mumbai,SQL,5,2,103,Rejected,45
3,3,Chirag,Bangalore,Python,3,3,102,Selected,92
4,3,Chirag,Bangalore,Python,3,8,104,Selected,90
5,4,Divya,Delhi,ML,7,5,101,Selected,85
6,5,Eshan,Mumbai,Python,1,6,102,Rejected,38
7,6,Fatima,Bangalore,SQL,4,7,103,Pending,67


# Part B — Groupby & Aggregation
Using the merged df:

## 1: Find the average InterviewScore per City

In [18]:
df.groupby("City")["InterviewScore"].mean()

City
Bangalore    83.0
Delhi        83.0
Mumbai       41.5
Name: InterviewScore, dtype: float64

## 2: Find the total number of applications per Job Title

In [20]:
df.groupby("Title")["AppID"].count()

Title
Data Analyst      2
ML Engineer       2
Python Dev        2
SQL Specialist    2
Name: AppID, dtype: int64

## 3: Find the highest InterviewScore per Status (Selected / Rejected / Pending)

In [21]:
df.groupby("Status")["InterviewScore"].max()

Status
Pending     76
Rejected    45
Selected    92
Name: InterviewScore, dtype: int64

## 4: Use .agg() to get mean, max, and min of InterviewScore grouped by Skills

In [23]:
df.groupby("Skills")["InterviewScore"].agg(["mean", "max", "min"])

,mean,max,min
Skills,,,
ML,85.0,85,85
Python,76.8,92,38
SQL,56.0,67,45


## 5: Use .groupby() with multiple columns — find average InterviewScore grouped by both City and Status

In [24]:
df.groupby(["City", "Status"])["InterviewScore"].mean()

City       Status  
Bangalore  Pending     67.0
           Selected    91.0
Delhi      Pending     76.0
           Selected    86.5
Mumbai     Rejected    41.5
Name: InterviewScore, dtype: float64

# Part C — Transform & Filter

## 1: Use .transform() to add a column CityAvgScore — the average interview score for each candidate's city

In [25]:
df["CityAvgScore"] = df.groupby("City")["InterviewScore"].transform("mean")
df

,AppID,CandidateID,JobID,Status,InterviewScore,Name,City,Skills,YearsExp,Title,MinExp,Location,CityAvgScore
0,1,1,101,Selected,88,Aarav,Delhi,Python,2,Data Analyst,2,Delhi,83.0
1,2,2,103,Rejected,45,Bhavna,Mumbai,SQL,5,Python Dev,2,Mumbai,41.5
2,3,3,102,Selected,92,Chirag,Bangalore,Python,3,ML Engineer,5,Bangalore,83.0
3,4,1,104,Pending,76,Aarav,Delhi,Python,2,SQL Specialist,3,Delhi,83.0
4,5,4,101,Selected,85,Divya,Delhi,ML,7,Data Analyst,2,Delhi,83.0
5,6,5,102,Rejected,38,Eshan,Mumbai,Python,1,ML Engineer,5,Bangalore,41.5
6,7,6,103,Pending,67,Fatima,Bangalore,SQL,4,Python Dev,2,Mumbai,83.0
7,8,3,104,Selected,90,Chirag,Bangalore,Python,3,SQL Specialist,3,Delhi,83.0


## 2: Add a column AboveAvg — True if the candidate's InterviewScore is above their city average, False otherwise

In [27]:
df["AboveAvg"] = df["InterviewScore"] > df["CityAvgScore"]
df

,AppID,CandidateID,JobID,Status,InterviewScore,Name,City,Skills,YearsExp,Title,MinExp,Location,CityAvgScore,AboveAvg
0,1,1,101,Selected,88,Aarav,Delhi,Python,2,Data Analyst,2,Delhi,83.0,True
1,2,2,103,Rejected,45,Bhavna,Mumbai,SQL,5,Python Dev,2,Mumbai,41.5,True
2,3,3,102,Selected,92,Chirag,Bangalore,Python,3,ML Engineer,5,Bangalore,83.0,True
3,4,1,104,Pending,76,Aarav,Delhi,Python,2,SQL Specialist,3,Delhi,83.0,False
4,5,4,101,Selected,85,Divya,Delhi,ML,7,Data Analyst,2,Delhi,83.0,True
5,6,5,102,Rejected,38,Eshan,Mumbai,Python,1,ML Engineer,5,Bangalore,41.5,False
6,7,6,103,Pending,67,Fatima,Bangalore,SQL,4,Python Dev,2,Mumbai,83.0,False
7,8,3,104,Selected,90,Chirag,Bangalore,Python,3,SQL Specialist,3,Delhi,83.0,True


## 3: Use .filter() to keep only cities where the average InterviewScore is above 70

In [30]:
df.groupby("City").filter(lambda x: x["InterviewScore"].mean() > 70)

,AppID,CandidateID,JobID,Status,InterviewScore,Name,City,Skills,YearsExp,Title,MinExp,Location,CityAvgScore,AboveAvg
0,1,1,101,Selected,88,Aarav,Delhi,Python,2,Data Analyst,2,Delhi,83.0,True
2,3,3,102,Selected,92,Chirag,Bangalore,Python,3,ML Engineer,5,Bangalore,83.0,True
3,4,1,104,Pending,76,Aarav,Delhi,Python,2,SQL Specialist,3,Delhi,83.0,False
4,5,4,101,Selected,85,Divya,Delhi,ML,7,Data Analyst,2,Delhi,83.0,True
6,7,6,103,Pending,67,Fatima,Bangalore,SQL,4,Python Dev,2,Mumbai,83.0,False
7,8,3,104,Selected,90,Chirag,Bangalore,Python,3,SQL Specialist,3,Delhi,83.0,True


## 4: Add a ScoreRank column using .rank(method='dense', ascending=False) on InterviewScore

In [31]:
df["ScoreRank"] = df["InterviewScore"].rank(method="dense", ascending=False)
df

,AppID,CandidateID,JobID,Status,InterviewScore,Name,City,Skills,YearsExp,Title,MinExp,Location,CityAvgScore,AboveAvg,ScoreRank
0,1,1,101,Selected,88,Aarav,Delhi,Python,2,Data Analyst,2,Delhi,83.0,True,3.0
1,2,2,103,Rejected,45,Bhavna,Mumbai,SQL,5,Python Dev,2,Mumbai,41.5,True,7.0
2,3,3,102,Selected,92,Chirag,Bangalore,Python,3,ML Engineer,5,Bangalore,83.0,True,1.0
3,4,1,104,Pending,76,Aarav,Delhi,Python,2,SQL Specialist,3,Delhi,83.0,False,5.0
4,5,4,101,Selected,85,Divya,Delhi,ML,7,Data Analyst,2,Delhi,83.0,True,4.0
5,6,5,102,Rejected,38,Eshan,Mumbai,Python,1,ML Engineer,5,Bangalore,41.5,False,8.0
6,7,6,103,Pending,67,Fatima,Bangalore,SQL,4,Python Dev,2,Mumbai,83.0,False,6.0
7,8,3,104,Selected,90,Chirag,Bangalore,Python,3,SQL Specialist,3,Delhi,83.0,True,2.0


# Part D — String Operations & Type Conversion
Using the merged df:

## 1: Convert all names in Name to uppercase using .str.upper()

In [32]:
df["Name"].str.upper()

0     AARAV
1    BHAVNA
2    CHIRAG
3     AARAV
4     DIVYA
5     ESHAN
6    FATIMA
7    CHIRAG
Name: Name, dtype: object

## 2: Check which candidates have "Python" in their Skills column using .str.contains()

In [33]:
df["Skills"].str.contains("Python")

0     True
1    False
2     True
3     True
4    False
5     True
6    False
7     True
Name: Skills, dtype: bool

## 3: Add a column ExpLevel using .apply():

- YearsExp >= 5 → "Senior"
- YearsExp >= 3 → "Mid"
- Below 3 → "Junior"

In [36]:
df["ExpLevel"] = df["YearsExp"].apply(lambda x: "Senior" if x >= 5 else "Mid" if x >= 3 else "Junior")
df

,AppID,CandidateID,JobID,Status,InterviewScore,Name,City,Skills,YearsExp,Title,MinExp,Location,CityAvgScore,AboveAvg,ScoreRank,ExpLevel
0,1,1,101,Selected,88,Aarav,Delhi,Python,2,Data Analyst,2,Delhi,83.0,True,3.0,Junior
1,2,2,103,Rejected,45,Bhavna,Mumbai,SQL,5,Python Dev,2,Mumbai,41.5,True,7.0,Senior
2,3,3,102,Selected,92,Chirag,Bangalore,Python,3,ML Engineer,5,Bangalore,83.0,True,1.0,Mid
3,4,1,104,Pending,76,Aarav,Delhi,Python,2,SQL Specialist,3,Delhi,83.0,False,5.0,Junior
4,5,4,101,Selected,85,Divya,Delhi,ML,7,Data Analyst,2,Delhi,83.0,True,4.0,Senior
5,6,5,102,Rejected,38,Eshan,Mumbai,Python,1,ML Engineer,5,Bangalore,41.5,False,8.0,Junior
6,7,6,103,Pending,67,Fatima,Bangalore,SQL,4,Python Dev,2,Mumbai,83.0,False,6.0,Mid
7,8,3,104,Selected,90,Chirag,Bangalore,Python,3,SQL Specialist,3,Delhi,83.0,True,2.0,Mid


## 4: Convert InterviewScore to int32 using .astype()

In [43]:
df["InterviewScore"].astype("int32")

0    88
1    45
2    92
3    76
4    85
5    38
6    67
7    90
Name: InterviewScore, dtype: int32

## 5: Replace "Pending" in Status with "Under Review" using .replace()

In [45]:
df["Status"]= df["Status"].replace({"Pending": "Under Review"})
df

,AppID,CandidateID,JobID,Status,InterviewScore,Name,City,Skills,YearsExp,Title,MinExp,Location,CityAvgScore,AboveAvg,ScoreRank,ExpLevel
0,1,1,101,Selected,88,Aarav,Delhi,Python,2,Data Analyst,2,Delhi,83.0,True,3.0,Junior
1,2,2,103,Rejected,45,Bhavna,Mumbai,SQL,5,Python Dev,2,Mumbai,41.5,True,7.0,Senior
2,3,3,102,Selected,92,Chirag,Bangalore,Python,3,ML Engineer,5,Bangalore,83.0,True,1.0,Mid
3,4,1,104,Under Review,76,Aarav,Delhi,Python,2,SQL Specialist,3,Delhi,83.0,False,5.0,Junior
4,5,4,101,Selected,85,Divya,Delhi,ML,7,Data Analyst,2,Delhi,83.0,True,4.0,Senior
5,6,5,102,Rejected,38,Eshan,Mumbai,Python,1,ML Engineer,5,Bangalore,41.5,False,8.0,Junior
6,7,6,103,Under Review,67,Fatima,Bangalore,SQL,4,Python Dev,2,Mumbai,83.0,False,6.0,Mid
7,8,3,104,Selected,90,Chirag,Bangalore,Python,3,SQL Specialist,3,Delhi,83.0,True,2.0,Mid


# Part E — Full Recruitment Report
## Write a function recruitment_report(df) that prints the following summary — all values computed dynamically:

```
===== CodeHire Recruitment Report =====

Total Applications     : 8
Selected               : 3
Rejected               : 2
Under Review           : 3

Top Hiring City        : Delhi
Average Interview Score: 72.6

Best Performer         : Chirag (Score: 92)
Least Experienced Rep  : Eshan (1 year)

Jobs with no Selections: SQL Specialist

```
All values must come from the data — nothing hardcoded.

In [67]:
selected = df[df["Status"] == "Selected"]["Title"].unique()
total_jobs = df["Title"].unique()
not_selected = [title for title in total_jobs if title not in selected]
print(not_selected)

['Python Dev']


In [ ]:
def recruitment_report(df):
    total_application = df["AppId"].count()
    total_selected = (df["Status"] == 'Selected').sum()
    total_under_review = (df["Status"] == 'Under Review').sum()
    top_hiring_city = df.groupby("City")["Status"].count().idxmax()
    average_intervier_score = df["InterviewScore"].mean()
    best_performer = df.groupby("Name")["InterviewScore"].max().idxmax()
    best_performers_score = df.groupby("Name")["InterviewScore"].max().max()
    least_performer = df.groupby("Name")["InterviewScore"].min().idxmin()
    least_performers_score = df.groupby("Name")["InterviewScore"].min().min()
    selected = df[df["Status"] == "Selected"]["Title"].unique()
    total_jobs = df["Title"].unique()
    not_selected = [title for title in total_jobs if title not in selected]

    print(f"Total Applications  : {total_applications}")
    print(f"Selected            : {total_selected}")
    print(f"Under Review        : {total_under_review}")
    print(f"Top Hiring City     : {top_hiring_city}")
    print(f"Avg Interview Score : {average_intervier_score}")
    print(f"Best Performer      : {best_performer} ({best_performers_score})")
    print(f"Least Exp Rep       : {least_performer} ({least_performers_score}")
    print(f"Job with 0 selects  : {not}")
Selected               : 3
Rejected               : 2
Under Review           : 3

Top Hiring City        : Delhi
Average Interview Score: 72.6

Best Performer         : Chirag (Score: 92)
Least Experienced Rep  : Eshan (1 year)

Jobs with no Selections: SQL Specialist
    